# 01 — Understanding the BCRP API

Goal: learn how BCRPData's public API behaves **by probing it**, before writing any production code.

- Series: `PD04638PD` (interbank selling rate, S/ per US$) and `PD04637PD` (buying rate).
- Endpoint: `https://estadisticas.bcrp.gob.pe/estadisticas/series/api/{codes}/json/{start}/{end}/ing`

Everything useful found here migrates to `pipeline/src/fxcast/ingest/` with tests.

## 1. First request

Build the URL by hand and make a single `GET` with **httpx**. Before parsing anything, look at the *envelope* of the response:

- **Status code**: `200` means the server answered OK — not necessarily that the data is what we expect.
- **`Content-Type`**: tells the client how to interpret the body. Check it carefully: is it what a JSON API *should* send?
- **`server` / `x-cdn`**: who is actually answering? (Relevant later for retries and blocking.)

`timeout=30.0` is explicit on purpose: httpx's default is 5 s, and a production client must **never** rely on implicit timeouts.

We use a **fixed** date range so the output is reproducible when you re-run the notebook.

**Expect:** status `200`, a response time under ~2 s, and one surprise in the headers.

In [ ]:
import httpx

BASE_URL = "https://estadisticas.bcrp.gob.pe/estadisticas/series/api"
SERIES_SELL = "PD04638PD"  # Interbank selling rate (S/ per US$)

url = f"{BASE_URL}/{SERIES_SELL}/json/2026-11-02/2026-11-20/ing"
response = httpx.get(url, timeout=30.0)

print("URL:         ", response.url)
print("Status:      ", response.status_code, response.reason_phrase)
print("Content-Type:", response.headers.get("content-type"))
print("Elapsed:     ", response.elapsed)
print("Server / CDN:", response.headers.get("server"), "/", response.headers.get("x-cdn"))

URL:          https://estadisticas.bcrp.gob.pe/estadisticas/series/api/PD04638PD/json/2026-11-02/2026-11-20/ing
Status:       200 OK
Content-Type: text/html; charset=UTF-8
Elapsed:      0:00:00.296407
Server / CDN: srv / Imperva


## 2. Anatomy of the JSON

Now decode the body and map its structure **before** writing any parser:

- Which top-level keys exist?
- What does `config` describe, and how does it relate to `periods`?
- What *type* is each value inside `periods` (look at the quotes)?

**Expect:** keys `config` and `periods`; ~15 business days between 2–20 Nov 2026; dates like `"02.Nov.26"` and values as **strings**.

In [ ]:
payload = response.json()

print("Top-level keys:", list(payload))
print("config:", payload["config"])
print("Number of periods:", len(payload["periods"]))

payload["periods"][:3]

Top-level keys: ['config', 'periods']
config: {'title': 'Tipo de cambio', 'series': [{'name': 'Tipo de cambio - TC Interbancario (S/ por US$) - Venta', 'dec': '3'}]}
Number of periods: 15


[{'name': '02.Nov.26', 'values': ['3.46057142857143']},
 {'name': '03.Nov.26', 'values': ['3.46342857142857']},
 {'name': '04.Nov.26', 'values': ['3.44714285714286']}]

## 3. Parsing dates: `"02.Nov.26"`

The format maps to `strptime` directives as `%d.%b.%y` (day, abbreviated month, **2-digit year**).

Two traps to probe:

1. **2-digit year pivot.** Python (POSIX rule) maps `69–99 → 19xx` and `00–68 → 20xx`. Our data starts in 1997, so we must check both sides of the pivot.
2. **`%b` is locale-dependent.** It parses month names in the process's `LC_TIME` locale. On a machine/container with a Spanish locale, `"Ene"` would parse and `"Jan"` might not.

**Expect:** `"02.Jan.97"` → 1997 ✅, but look closely at `"31.Dec.68"` and `"01.Jan.69"`: one day apart in the string, how far apart in the result?

In [ ]:
from datetime import date, datetime


def parse_date_strptime(raw: str) -> date:
    return datetime.strptime(raw, "%d.%b.%y").date()


for raw in ["02.Nov.26", "02.Jan.97", "31.Dec.68", "01.Jan.69"]:
    print(f"{raw!r:>12} -> {parse_date_strptime(raw)}")

 '02.Nov.26' -> 2026-11-02
 '02.Jan.97' -> 1997-01-02
 '31.Dec.68' -> 2068-12-31
 '01.Jan.69' -> 1969-01-01


## 4. Parsing values: strings, `"n.d."` and *silent* failures

Values arrive as strings, and the BCRP uses `"n.d."` (*no disponible*) for missing data.

Two strategies:

- **Explicit:** a known set of missing markers → `NaN`; anything else must be a valid number or **raise**.
- **Convenient:** `pd.to_numeric(..., errors="coerce")` — turns *anything* unparseable into `NaN`.

We feed both a realistic value, `"n.d."`, a value with spaces, and a value with a **decimal comma** (a plausible format change on a Spanish-language site).

**Expect:** the explicit parser raises on `"3,41"`; `pd.to_numeric` quietly returns `NaN` for it — indistinguishable from a real missing value. Also note `config` said `"dec": "3"` while values have ~14 decimals: which one should we store?

In [ ]:
import math

import pandas as pd

MISSING_MARKERS = {"n.d."}


def parse_value(raw: str) -> float:
    if raw.strip() in MISSING_MARKERS:
        return math.nan
    return float(raw)  # raises ValueError on anything unexpected


raws = ["3.46057142857143", "n.d.", " 3.41 ", "3,41"]

print("Explicit parser:")
for raw in raws:
    try:
        print(f"  {raw!r:>20} -> {parse_value(raw)}")
    except ValueError as exc:
        print(f"  {raw!r:>20} -> ValueError: {exc}")

print("\npd.to_numeric(errors='coerce'):")
print(pd.to_numeric(pd.Series(raws), errors="coerce"))

Explicit parser:
    '3.46057142857143' -> 3.46057142857143
                'n.d.' -> nan
              ' 3.41 ' -> 3.41
                '3,41' -> ValueError: could not convert string to float: '3,41'

pd.to_numeric(errors='coerce'):
0    3.460571
1         NaN
2    3.410000
3         NaN
dtype: float64


## 5. Edge cases: probing the API's behavior

Documentation rarely covers edge cases, so we **ask the API directly**. A small `probe()` helper classifies each response as JSON or not-JSON and summarizes it.

Cases: weekend-only range, `start > end`, an **invalid series code**, an end date **in the future**, a range starting **before** the series exists, two series in one call, and the **full history** (to find out if there is a max range).

`time.sleep(2)` between calls: we are guests on a public API. Being polite is also practical — a WAF (remember `x-cdn: Imperva`) may react to bursts.

**Expect:** most cases return JSON, but at least one does not. For each row ask: *if my parser saw this, would it fail loudly, or produce wrong data quietly?*

In [ ]:
import time
from typing import Any


def probe(codes: str, start: str, end: str) -> dict[str, Any]:
    resp = httpx.get(f"{BASE_URL}/{codes}/json/{start}/{end}/ing", timeout=30.0)
    try:
        periods = resp.json()["periods"]
    except ValueError:  # JSONDecodeError is a subclass of ValueError
        return {"kind": "NOT JSON", "bytes": len(resp.content), "head": resp.text[:60]}
    return {
        "kind": "JSON",
        "n_periods": len(periods),
        "first": periods[0] if periods else None,
        "last": periods[-1] if periods else None,
    }


cases = {
    "weekend only": ("PD04638PD", "2026-11-21", "2026-11-22"),
    "start > end": ("PD04638PD", "2026-11-10", "2026-11-02"),
    "invalid code": ("PD99999PD", "2026-11-02", "2026-11-06"),
    "end in future": ("PD04638PD", "2026-11-16", "2026-12-04"),
    "before series start": ("PD04638PD", "1996-12-25", "1997-01-03"),
    "two series": ("PD04637PD-PD04638PD", "2026-11-02", "2026-11-03"),
    "full history (2 series)": ("PD04637PD-PD04638PD", "1997-01-01", "2026-11-20"),
}

results = {}
for name, args in cases.items():
    results[name] = probe(*args)
    time.sleep(2)

pd.DataFrame(results).T

,kind,n_periods,first,last,bytes,head
weekend only,JSON,0,None,None,NaN,NaN
start > end,JSON,0,None,None,NaN,NaN
invalid code,NOT JSON,NaN,NaN,NaN,155,"<!DOCTYPE html>\n<html>\n<head><script src=""/o..."
end in future,JSON,15,"{'name': '16.Nov.26', 'values': ['3.4135714285...","{'name': '04.Dec.26', 'values': ['n.d.']}",NaN,NaN
before series start,JSON,8,"{'name': '25.Dec.96', 'values': ['n.d.']}","{'name': '03.Jan.97', 'values': ['2.62']}",NaN,NaN
two series,JSON,2,"{'name': '02.Nov.26', 'values': ['3.4588571428...","{'name': '03.Nov.26', 'values': ['3.4618571428...",NaN,NaN
full history (2 series),JSON,7798,"{'name': '01.Jan.97', 'values': ['n.d.', 'n.d.']}","{'name': '20.Nov.26', 'values': ['n.d.', 'n.d.']}",NaN,NaN


## 6. Full history: what the real data looks like

Load the full history of both series once and audit it **before** trusting it:

- Which **month tokens** appear? (Compare with the `%b` English abbreviations: `Jan Feb Mar Apr May Jun Jul Aug Sep Oct Nov Dec`.)
- How many `"n.d."`? Do the buying and selling series go missing **on the same days**?
- What do the **first** and **last** rows look like?

**Expect:** ~7,800 periods since Jan 1997, a few hundred `n.d.` rows, and one month token that would make `strptime("%d.%b.%y")` crash.

In [ ]:
from collections import Counter

full = httpx.get(
    f"{BASE_URL}/PD04637PD-PD04638PD/json/1997-01-01/2026-11-20/ing", timeout=60.0
).json()
periods = full["periods"]

month_tokens = sorted({p["name"].split(".")[1] for p in periods})
nd_pattern = Counter(tuple(v == "n.d." for v in p["values"]) for p in periods)

print("Series:", [s["name"] for s in full["config"]["series"]])
print("Periods:", len(periods))
print("Month tokens:", month_tokens)
print("n.d. pattern (buy is n.d., sell is n.d.) -> count:", dict(nd_pattern))
print("First 2:", periods[:2])
print("Last 2: ", periods[-2:])

Series: ['Tipo de cambio - TC Interbancario (S/ por US$) - Compra', 'Tipo de cambio - TC Interbancario (S/ por US$) - Venta']
Periods: 7798
Month tokens: ['Apr', 'Aug', 'Dec', 'Feb', 'Jan', 'Jul', 'Jun', 'Mar', 'May', 'Nov', 'Oct', 'Set']
n.d. pattern (buy is n.d., sell is n.d.) -> count: {(True, True): 355, (False, False): 7443}
First 2: [{'name': '01.Jan.97', 'values': ['n.d.', 'n.d.']}, {'name': '02.Jan.97', 'values': ['2.613', '2.618']}]
Last 2:  [{'name': '19.Nov.26', 'values': ['3.43914285714286', '3.44128571428571']}, {'name': '20.Nov.26', 'values': ['n.d.', 'n.d.']}]


## Key takeaways

**What the API actually does** (verified by probing, Nov 2026):

| Behavior | Evidence | Impact |
|---|---|---|
| Body is JSON but `Content-Type: text/html` | cell 1 | Never trust the header; decode and **validate structure** |
| Invalid series code → `200` + 155 B HTML page | cell 5 | Same page as a WAF (Imperva) challenge → ambiguous; retry, then fail with a clear message |
| Returns the **Mon–Fri grid** of the requested range, holidays and future dates as `"n.d."` | cells 5–6 | Two kinds of `n.d.`: interior (holiday → keep NaN) vs leading/trailing (not published → drop) |
| Last published value lags >1 business day | cell 6 (`20.Nov.26` = `n.d.` on Sun 22) | Incremental ingest re-fetches a window and upserts |
| September is `"Set"` even with `/ing` | cell 6 | `strptime("%b")` crashes → own month map |
| 2-digit years; Python pivots `68 → 2068`, `69 → 1969` | cell 3 | Resolve century inside the requested `[start, end]` |
| No max range: 1997–today, 2 series, 1 call (~0.5 s) | cell 5 | Backfill = one request, no chunking |
| Multi-series `values` are **positional** | cell 5 | Check `len(values) == len(config.series) == len(codes)` |
| Buy/sell are `n.d.` on exactly the same days (355) | cell 6 | Data-quality check if they ever diverge |
| `pd.to_numeric(errors="coerce")` hides format changes | cell 4 | Explicit parser: known markers → NaN, anything else raises |

**What migrates to `pipeline/src/fxcast/`:**

- `ingest/parse.py` — pure functions, no I/O: `parse_date`, `parse_value`, `parse_response` (shape checks, trim leading/trailing `n.d.`) → tested with recorded fixtures.
- `ingest/client.py` — httpx client: explicit timeout, retries with exponential backoff + jitter on transient errors (network, 5xx, not-JSON), generic User-Agent.
- `ingest/series.py` — series codes as constants (`PD04637PD` buy, `PD04638PD` sell).
- `storage/` — `Storage` protocol + Parquet implementation with idempotent upsert by date.

**Stays here:** `probe()` and the ad-hoc audits — exploration tools, not production code.